<a href="https://colab.research.google.com/github/hosseinta2/LLM-from-scratch/blob/main/LLM_from_scratch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import os
print(os.listdir())

['.config', 'story.txt', 'sample_data']


In [19]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Fri May 22 17:59:27 2026

@author: hosseintaheri
"""
import re
with open("story.txt", "r",encoding = "utf-8") as f:
    file = f.read()
print("number of chars:", len(file))

sample = file[:100]
print(sample)

result = re.split(r'([,.?_!"()\']|--|\s)', file) # tokenzed input
result = [x for x in result if x!=' ']
print(len(result))
print(result[:30])

all_words = sorted(list(set(result)))
vocab_size = len(all_words)
print(vocab_size)


vocab = {word:num for num,word in enumerate(all_words) } # for encoding

ids = [vocab[x] for x in result] # index for each token
print(ids[:20])

## final code

class tokenizer:
    def __init__(self, vocab):
        self.encoder = vocab
        self.decoder ={num:word for word,num in vocab.items()}
    def encode(self, text):
        tokenized = re.split(r'([,.?_!"()\']|--|\s)', text)
        tokenized = [x for x in tokenized if x!= ' ']
        ids =  [self.encoder[x] for x in tokenized]
        return ids
    def decode(self, ids):
        text = " ".join(self.decoder[x] for x in ids)
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

tkz = tokenizer(vocab)
result1 = tkz.encode(file)
print(result1[:10])
result2 = tkz.decode(result1)
print(result2[:1000])


number of chars: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no g
5598
['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', '']
1161
[57, 48, 156, 1030, 61, 41, 841, 121, 265, 496, 8, 1029, 121, 510, 445, 401, 8, 935, 598, 1106]
[57, 48, 156, 1030, 61, 41, 841, 121, 265, 496]
I HAD always thought Jack Gisburn rather a cheap genius -- though a good fellow enough -- so it was no great surprise to me to hear that,  in the height of his glory,  he had dropped his painting,  married a rich widow,  and established himself in a villa on the Riviera.( Though I rather thought it would have been Rome or Florence.)" The height of his glory"  -- that was what the women called it.  I can hear Mrs.  Gideon Thwing -- his last Chicago sitter -- deploring his unaccountable a

adding uknown tokens and end of text tokens to the vocab list and updating encoder

In [30]:
all_words = all_words + ["<|endoftext|>", "<|unk|>"]
vocab = {word:num for num,word in enumerate(all_words)}
class tokenizer:
    def __init__(self, vocab):
        self.encoder = vocab
        self.decoder ={num:word for word,num in vocab.items()}
    def encode(self, text):
        tokenized = re.split(r'([,.?_!"()\']|--|\s)', text)
        tokenized = [x for x in tokenized if x!= ' ']
        tokenized = [x if x in self.encoder else '<|unk|>' for x in tokenized ]
        ids =  [self.encoder[x] for x in tokenized]
        return ids
    def decode(self, ids):
        text = " ".join(self.decoder[x] for x in ids)
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text
tkz = tokenizer(vocab)
print(tkz.decode(tkz.encode("Hello, my beloved friend, I had always thought of you <|endoftext|>")))


<|unk|>,  my <|unk|> friend,  I had always thought of you <|endoftext|>


Implementing Byte pair token

In [31]:
pip install tiktoken

In [32]:
from importlib.metadata import version

import tiktoken
print("tiktoken version:", version("tiktoken"))

tiktoken version: 0.12.0


In [33]:
tokenizer = tiktoken.get_encoding("gpt2")

In [54]:
ids = tokenizer.encode(file)
print(ids[:10])

[40, 367, 2885, 1464, 1807, 3619, 402, 271, 10899, 2138]


In [56]:
print(tokenizer.decode(ids)[:10])
print(len(ids))

I HAD alwa
5145


In [59]:
context_size = 10
for i in range(1,context_size):
  x = ids[:i]
  y = [ids[i]]
  print(tokenizer.decode(x),tokenizer.decode(y))

I  H
I H AD
I HAD  always
I HAD always  thought
I HAD always thought  Jack
I HAD always thought Jack  G
I HAD always thought Jack G is
I HAD always thought Jack Gis burn
I HAD always thought Jack Gisburn  rather
